# Undo-first exploration for ARC-AGI-3

**Reversibility-aware frontier exploration**: a training-free, LLM-free agent that builds a transition graph
of each level, learns which actions can be undone (inverse pairs, the `ACTION7` undo, a learned UI-counter mask),
tests undoable actions first and routes to unexplored states through edges it knows how to reverse.

This notebook is the code companion of the ARC Prize 2026 paper-track writeup *Undo-first exploration:
reversibility-aware frontier search for ARC-AGI-3*. It runs fully offline on the 25 public games with the
official `arc-agi` toolkit (Python 3.12) and scores with the toolkit's own formula
(`EnvironmentScoreCalculator`: level score = min((human/agent)^2 x 100, 115), level-index-weighted game score).

Three modes are compared with identical code paths:

| mode | what it does |
|---|---|
| `random` | uniform random over the available actions (the Kaggle starter baseline) |
| `frontier` | graph exploration to the nearest untested state-action pair (prior-work style) |
| `undo` | frontier + reversibility ordering, inverse-pair learning, undo verification, irreversible-edge path penalty |

No LLM, no network, no game-specific code, no exploitation of the null-coordinate crash reported by Liew (2026).

In [ ]:
import os, sys, glob, subprocess, tempfile
KAGGLE_WHEELS = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
KAGGLE_ENVS = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
try:
    import arc_agi, arcengine  # noqa
    print("arc-agi", arc_agi.__version__ if hasattr(arc_agi, "__version__") else "present")
except ImportError:
    if os.path.isdir(KAGGLE_WHEELS):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", KAGGLE_WHEELS, "arc-agi", "python-dotenv"], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arc-agi"], check=True)
# where the 25 public game files live: the Kaggle competition bundle, ../games/environment_files locally,
# or (anywhere with internet) a one-time download through the toolkit's anonymous API key
if os.path.isdir(KAGGLE_ENVS):
    os.environ["ARC_ENV_DIR"] = KAGGLE_ENVS
elif os.path.isdir("../games/environment_files"):
    os.environ["ARC_ENV_DIR"] = os.path.abspath("../games/environment_files")
else:
    os.environ["ARC_ENV_DIR"] = os.path.join(tempfile.gettempdir(), "arc_environment_files")  # outside the notebook's outputs
    if not os.path.isdir(os.environ["ARC_ENV_DIR"]):
        from arc_agi import Arcade, OperationMode
        arc = Arcade(operation_mode=OperationMode.NORMAL, environments_dir=os.environ["ARC_ENV_DIR"])
        for e in arc.get_environments():
            arc.make(e.game_id)          # downloads and caches the game source
games = [d for d in os.listdir(os.environ["ARC_ENV_DIR"]) if not d.startswith(".")]
assert len(games) == 25, games
print("games:", len(games), "in", os.environ["ARC_ENV_DIR"])

## The agent (single file, numpy only)

In [ ]:
%%writefile undo_agent.py
"""Reversibility-aware frontier exploration for ARC-AGI-3 (training-free, LLM-free, numpy only).

Modes (same code path, flags differ):
  random   : uniform random over the available actions (the Kaggle starter baseline)
  frontier : transition graph per level; untested actions first (ordered by a learned responsiveness
             prior and by action types seen on earlier winning paths); Dijkstra to the nearest frontier
  undo     : frontier + reversibility: a learned per-action-type reversibility rate orders probes
             (likely-undoable first), frontier choice adds the expected cost of getting back
             (RESET + replay) for a probe that cannot be undone, routes prefer edges with a known
             way back, and ACTION7 is verified as an undo early and then trusted.
"""
from __future__ import annotations

import hashlib
import heapq
import random
from collections import deque
from typing import Any

import numpy as np

try:
    from arcengine import GameAction, GameState
except ImportError:  # pragma: no cover - the Kaggle framework ships arcengine
    GameAction = GameState = None  # type: ignore

NOOP, OVER = "noop", "over"
UNDO_ID, CLICK_ID, RESET_ID = 7, 6, 0
EDGE_ROWS = 4            # UI strips (step counter, level indicator) live in the top/bottom rows
IRREVERSIBLE_PENALTY = 1.0


def frame_key(fd, mask: np.ndarray | None = None) -> tuple[bytes, np.ndarray] | None:
    """Hash of the last rendered frame with the learned UI cells zeroed; None when no frame was sent."""
    fr = getattr(fd, "frame", None)
    if not fr:
        return None
    arr = np.asarray(fr[-1], dtype=np.uint8)
    view = arr if mask is None else np.where(mask, 0, arr)
    return hashlib.blake2b(view.tobytes(), digest_size=8).digest(), arr


def click_targets(arr: np.ndarray, cap: int = 40) -> list[tuple[int, int, int]]:
    """One click target (x, y, colour) per 4-connected same-colour component, largest first.
    The component covering >40% of the grid is background. ponytail: pure-Python flood fill,
    a few ms per new state; vectorise if profiling says so."""
    h, w = arr.shape
    seen = np.zeros((h, w), dtype=bool)
    comps: list[tuple[int, tuple[int, int, int]]] = []
    for y0 in range(h):
        for x0 in range(w):
            if seen[y0, x0]:
                continue
            c = int(arr[y0, x0])
            stack = [(y0, x0)]
            seen[y0, x0] = True
            cells = []
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and arr[ny, nx] == c:
                        seen[ny, nx] = True
                        stack.append((ny, nx))
            n = len(cells)
            if n > 0.4 * h * w:
                continue
            cy = sum(p[0] for p in cells) / n
            cx = sum(p[1] for p in cells) / n
            y, x = min(cells, key=lambda p: (p[0] - cy) ** 2 + (p[1] - cx) ** 2)
            comps.append((n, (int(x), int(y), c)))
    comps.sort(key=lambda t: -t[0])
    return [xyc for _, xyc in comps[:cap]]


def atype(a: Any) -> Any:
    """Action type used for the learned priors: the action id, or ('c', colour) for a click."""
    return ("c", a[3]) if isinstance(a, tuple) else a


class UndoAgent:
    """Duck-typed on the ARC-AGI-3 agent interface: choose_action(frames, latest_frame) -> GameAction."""

    MAX_ACTIONS = 10000

    def __init__(self, mode: str = "undo", seed: int = 0, max_actions: int | None = None):
        assert mode in ("random", "frontier", "undo")
        self.mode = mode
        self.rng = random.Random(seed)
        if max_actions:
            self.MAX_ACTIONS = max_actions
        # ---- game-level priors (survive level changes) ----
        self.tested: dict[Any, int] = {}     # action type -> transitions tested
        self.changed: dict[Any, int] = {}    # action type -> transitions that changed the state
        self.reversed_: dict[Any, int] = {}  # action type -> state-changing transitions with a known way back
        self.inverse: dict[int, int] = {}    # action id -> learned inverse id (keyboard games)
        self.undo_works: bool | None = None  # ACTION7 verified to return to the previous state
        self.win_types: set[Any] = set()     # action types on earlier levels' winning paths
        self.mask = np.zeros((64, 64), dtype=bool)
        self.prev_arr: np.ndarray | None = None
        # ---- bookkeeping ----
        self.levels_done = 0
        self.level_actions: list[int] = []   # actions spent on each completed level (RESETs included)
        self.actions_this_level = 0
        self.action_counter = 0
        self.pending_data: dict[str, Any] = {}
        self.stats = {"reversible_edges": 0, "irreversible_edges": 0, "noop_edges": 0, "fatal_edges": 0,
                      "states": 0, "game_overs": 0, "resets": 0, "plans": 0}
        self.nodes: dict[bytes, dict[Any, Any]] = {}
        self.rev_edges: set[tuple[bytes, Any]] = set()
        self._new_level()

    # ----- per-level state ------------------------------------------------------
    def close_level(self) -> None:
        """Fold the finished level's graph into the run statistics (final reversibility classes)."""
        self.stats["states"] += len(self.nodes)
        for s, node in self.nodes.items():
            for a, r in node.items():
                if r is None:
                    continue
                k = "noop_edges" if r == NOOP else "fatal_edges" if r == OVER else \
                    "reversible_edges" if (s, a) in self.rev_edges else "irreversible_edges"
                self.stats[k] += 1

    def _new_level(self) -> None:
        self.close_level()
        self.nodes = {}
        self.rev_edges = set()
        self.start: bytes | None = None
        self.prev: tuple[bytes, Any] | None = None          # (state, action) just taken
        self.prev_change: tuple[bytes, Any] | None = None   # last transition that changed the state
        self.plan: deque[tuple[bytes, Any]] = deque()       # (state to be in, action to take) ...

    def _action_keys(self, fd, arr: np.ndarray) -> list[Any]:
        keys: list[Any] = []
        for aid in fd.available_actions:
            if aid == RESET_ID:
                continue
            if aid == CLICK_ID:
                keys.extend(("c",) + t for t in click_targets(arr))
            else:
                keys.append(aid)
        return keys

    # ----- interface --------------------------------------------------------------
    def is_done(self, frames, latest_frame) -> bool:
        return latest_frame.state == GameState.WIN or self.action_counter >= self.MAX_ACTIONS

    def choose_action(self, frames, latest):
        self.action_counter += 1
        self.actions_this_level += 1
        if latest.levels_completed > self.levels_done:          # the previous action finished a level
            self.level_actions.append(self.actions_this_level - 1)
            self.levels_done = latest.levels_completed
            self.actions_this_level = 1
            self._remember_winning_path()
            self._new_level()

        if latest.state == GameState.GAME_OVER:
            if self.prev:
                self.nodes[self.prev[0]][self.prev[1]] = OVER
            self.stats["game_overs"] += 1
            self.prev = self.prev_change = None
            self.plan.clear()
            return self._emit(RESET_ID)
        if latest.state == GameState.NOT_PLAYED:
            return self._emit(RESET_ID)

        fk = frame_key(latest, self.mask)
        if fk is None:
            return self._emit(RESET_ID)
        key, arr = fk
        if self._learn_mask(arr):                      # mask grew: old hashes are stale, restart the level graph
            key = frame_key(latest, self.mask)[0]
            self._new_level()
        self.prev_arr = arr
        if key not in self.nodes:
            self.nodes[key] = {a: None for a in self._action_keys(latest, arr)}
        if self.start is None:
            self.start = key

        self._record(key)
        akey = self._pick(key)
        self.prev = (key, akey) if akey != RESET_ID else None
        if akey == RESET_ID:
            self.prev_change = None
            self.plan.clear()
        return self._emit(akey)

    # ----- learning ---------------------------------------------------------------
    def _learn_mask(self, arr: np.ndarray) -> bool:
        """An edge row (top/bottom EDGE_ROWS) that changes by <= 2 cells in one transition is a UI
        counter bar, not game state: mask the whole row. ponytail: row-level, edge-only; learn
        arbitrary regions if a private game draws its counter elsewhere."""
        if self.prev_arr is None or self.prev_arr.shape != arr.shape:
            return False
        h = arr.shape[0]
        changed = np.argwhere(self.prev_arr != arr)
        grew = False
        for r in np.unique(changed[:, 0]):
            if (r < EDGE_ROWS or r >= h - EDGE_ROWS) and not self.mask[r].all() and (changed[:, 0] == r).sum() <= 2:
                self.mask[r] = True
                grew = True
        return grew

    def _record(self, key: bytes) -> None:
        """Classify the transition that led here and update the per-type priors."""
        if not self.prev:
            return
        s, a = self.prev
        t = atype(a)
        self.tested[t] = self.tested.get(t, 0) + 1
        if key == s:
            self.nodes[s][a] = NOOP
            if a == UNDO_ID and self.prev_change and self.undo_works is None:
                self.undo_works = False               # had something to undo, nothing happened
            return
        self.nodes[s][a] = key
        self.changed[t] = self.changed.get(t, 0) + 1
        # every known edge key -> s is now reversible (we just found a way back from s to key... no:
        # we found s -> key; so edges key -> s that were already known gain a way back)
        for b, r in self.nodes.get(key, {}).items():
            if r == s and (key, b) not in self.rev_edges:
                self.rev_edges.add((key, b))
                tb = atype(b)
                self.reversed_[tb] = self.reversed_.get(tb, 0) + 1
        if s in self.nodes.get(key, {}).values() and (s, a) not in self.rev_edges:
            self.rev_edges.add((s, a))
            self.reversed_[t] = self.reversed_.get(t, 0) + 1
        if self.prev_change and self.prev_change[0] == key:     # s_prev --a0--> s --a--> s_prev
            a0 = self.prev_change[1]
            if a == UNDO_ID:
                self.undo_works = True
            elif isinstance(a, int) and isinstance(a0, int) and UNDO_ID not in (a, a0):
                self.inverse[a0], self.inverse[a] = a, a0
        elif a == UNDO_ID and self.prev_change and self.undo_works is None:
            self.undo_works = False                   # undo went somewhere else
        self.prev_change = (s, a)

    def _remember_winning_path(self) -> None:
        """Action types on the shortest known path start -> winning transition get priority next level."""
        if not self.prev or self.start is None:
            return
        goal, last = self.prev
        self.win_types.add(atype(last))
        parent: dict[bytes, tuple[bytes, Any]] = {}
        q = deque([self.start])
        seen = {self.start}
        while q:
            s = q.popleft()
            if s == goal:
                break
            for a, r in self.nodes.get(s, {}).items():
                if r in (None, NOOP, OVER) or r in seen:
                    continue
                seen.add(r)
                parent[r] = (s, a)
                q.append(r)
        s = goal
        while s in parent:
            s, a = parent[s]
            self.win_types.add(atype(a))

    # ----- priors ------------------------------------------------------------------
    def p_change(self, a: Any) -> float:
        t = atype(a)
        return (self.changed.get(t, 0) + 1) / (self.tested.get(t, 0) + 2)

    def p_rev(self, a: Any) -> float:
        if a == UNDO_ID:
            return 0.0 if self.undo_works is False else 1.0
        if self.undo_works:
            return 1.0
        if isinstance(a, int) and a in self.inverse:
            return 1.0
        t = atype(a)
        return (self.reversed_.get(t, 0) + 1) / (self.changed.get(t, 0) + 2)

    def _order_key(self, a: Any) -> tuple:
        k: list[Any] = [0 if atype(a) in self.win_types else 1]
        if self.mode == "undo":
            k.append(-round(self.p_rev(a), 2))
        k.append(-round(self.p_change(a), 2))
        k.append(self.rng.random())
        return tuple(k)

    # ----- policy ------------------------------------------------------------------
    def _pick(self, key: bytes) -> Any:
        node = self.nodes[key]
        if self.mode == "random":
            aids = list({k if isinstance(k, int) else CLICK_ID for k in node})
            if not aids:
                return RESET_ID
            a = self.rng.choice(aids)
            return ("c", self.rng.randrange(64), self.rng.randrange(64), -1) if a == CLICK_ID else a
        untested = [a for a, r in node.items() if r is None]
        if untested:
            self.plan.clear()
            return min(untested, key=self._order_key)
        if self.plan and self.plan[0][0] == key:        # keep following the cached route
            return self.plan.popleft()[1]
        self.plan.clear()
        route = self._route_to_frontier(key)
        if route:
            self.stats["plans"] += 1
            self.plan = deque(route)
            return self.plan.popleft()[1]
        if key != self.start and self.start in self.nodes and any(r is None for r in self.nodes[self.start].values()):
            self.stats["resets"] += 1
            return RESET_ID
        live = [a for a, r in node.items() if r not in (NOOP, OVER, None)]   # graph exhausted: keep the game alive
        return self.rng.choice(live) if live else RESET_ID

    def _dijkstra(self, src: bytes) -> tuple[dict[bytes, float], dict[bytes, tuple[bytes, Any]]]:
        dist = {src: 0.0}
        parent: dict[bytes, tuple[bytes, Any]] = {}
        heap = [(0.0, 0, src)]
        tie = 0
        while heap:
            d, _, s = heapq.heappop(heap)
            if d > dist.get(s, float("inf")):
                continue
            for a, r in self.nodes.get(s, {}).items():
                if r in (None, NOOP, OVER):
                    continue
                w = 1.0 + (IRREVERSIBLE_PENALTY if self.mode == "undo" and (s, a) not in self.rev_edges else 0.0)
                nd = d + w
                if nd < dist.get(r, float("inf")):
                    dist[r] = nd
                    parent[r] = (s, a)
                    tie += 1
                    heapq.heappush(heap, (nd, tie, r))
        return dist, parent

    def _route_to_frontier(self, src: bytes) -> list[tuple[bytes, Any]] | None:
        """Cheapest frontier: travel cost, plus (undo mode) the expected cost of getting back after the
        probe we would make there: (1 - p_rev) x (RESET + replay from the level start)."""
        dist, parent = self._dijkstra(src)
        from_start = self._dijkstra(self.start)[0] if (self.mode == "undo" and self.start is not None) else {}
        best, best_cost = None, float("inf")
        for t, d in dist.items():
            if t == src:
                continue
            untested = [a for a, r in self.nodes.get(t, {}).items() if r is None]
            if not untested:
                continue
            cost = d
            if self.mode == "undo":
                a = min(untested, key=self._order_key)
                cost += (1.0 - self.p_rev(a)) * (1.0 + from_start.get(t, d))
            if cost < best_cost:
                best, best_cost = t, cost
        if best is None:
            return None
        route: list[tuple[bytes, Any]] = []
        s = best
        while s != src:
            ps, a = parent[s]
            route.append((ps, a))
            s = ps
        route.reverse()
        return route

    def _emit(self, akey: Any):
        if isinstance(akey, tuple):
            act = GameAction.ACTION6
            self.pending_data = {"x": akey[1], "y": akey[2]}
        else:
            act = GameAction.from_id(akey)
            self.pending_data = {}
        act.set_data(self.pending_data) if act.is_complex() else act.set_data({})
        return act


## Local runner and the official score formula

In [ ]:
%%writefile run_local.py
"""Run an agent mode over the public ARC-AGI-3 games offline and score with the official formula.

usage: python run_local.py --mode undo --seed 0 --max-actions 3000 --games ls20,vc33 --out results/undo-s0.json
"""
import argparse, json, logging, os, sys, time
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
logging.disable(logging.INFO)

ENV_DIR = os.environ.get("ARC_ENV_DIR") or os.path.join(os.path.dirname(os.path.abspath(__file__)), "games", "environment_files")


def level_score(baseline: int, actions: int) -> float:
    """arc_agi.scorecard.EnvironmentScoreCalculator: ((baseline/actions)^2*100, capped at 115)."""
    return min((baseline / actions) ** 2 * 100, 115.0) if actions > 0 else 0.0


def game_score(baselines: list[int], level_actions: list[int]) -> float:
    """Weighted by 1-indexed level number; unfinished levels score 0; capped by the finished-weight share."""
    tot = sum(range(1, len(baselines) + 1))
    done_w = sum(i + 1 for i in range(len(level_actions)))
    s = sum((i + 1) * level_score(b, a) for i, (b, a) in enumerate(zip(baselines, level_actions))) / tot
    return min(s, done_w / tot * 100)


def play(args):
    game_id, mode, seed, max_actions = args
    from arc_agi import Arcade, OperationMode
    from undo_agent import UndoAgent
    arc = Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=ENV_DIR)
    info = next(e for e in arc.get_environments() if e.game_id.startswith(game_id))
    env = arc.make(info.game_id)
    fd = env.reset()
    agent = UndoAgent(mode=mode, seed=seed, max_actions=max_actions)
    t0 = time.time()
    n = 0
    while fd is not None and not agent.is_done([fd], fd):
        act = agent.choose_action([fd], fd)
        fd = env.step(act, data=agent.pending_data)
        n += 1
    levels = fd.levels_completed if fd is not None else agent.levels_done
    if fd is not None and levels > agent.levels_done:          # last action finished a level
        agent.level_actions.append(agent.actions_this_level)
    agent.close_level()
    base = info.baseline_actions or []
    return {
        "game": game_id, "mode": mode, "seed": seed, "actions": n, "levels": levels,
        "win_levels": fd.win_levels if fd is not None else len(base),
        "state": fd.state.name if fd is not None else "ERROR",
        "level_actions": agent.level_actions, "baseline_actions": base,
        "score": game_score(base, agent.level_actions), "seconds": round(time.time() - t0, 1),
        "stats": agent.stats, "undo_works": agent.undo_works, "inverse_pairs": len(agent.inverse) // 2,
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--mode", default="undo")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max-actions", type=int, default=10000)
    p.add_argument("--games", default="")
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--out", default="")
    a = p.parse_args()
    games = a.games.split(",") if a.games else sorted(d for d in os.listdir(ENV_DIR) if not d.startswith("."))
    jobs = [(g, a.mode, a.seed, a.max_actions) for g in games]
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        rows = list(ex.map(play, jobs))
    total = sum(r["score"] for r in rows) / len(rows)
    for r in rows:
        print(f"{r['game']:6s} levels {r['levels']}/{r['win_levels']}  score {r['score']:6.2f}  actions {r['actions']:5d}  "
              f"per-level {r['level_actions']}  {r['seconds']}s  inv={r['inverse_pairs']} undo={r['undo_works']} {r['stats']}")
    print(f"TOTAL {a.mode} seed {a.seed}: score {total:.2f}  levels {sum(r['levels'] for r in rows)}/{sum(r['win_levels'] for r in rows)}")
    if a.out:
        os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
        json.dump({"mode": a.mode, "seed": a.seed, "max_actions": a.max_actions, "total": total, "games": rows},
                  open(a.out, "w"), indent=1)


if __name__ == "__main__":
    main()


## Play all 25 public games in the three modes

Budget: 10,000 actions per game, seed 0 (the paper reports the median of seeds 0-2). Each mode takes a few minutes on
8 CPU workers; the engine runs at roughly 500-2000 steps per second.

In [ ]:
!python run_local.py --mode random --seed 0 --max-actions 10000 --workers 8 --out results/random-s0.json | tail -3

In [ ]:
!python run_local.py --mode frontier --seed 0 --max-actions 10000 --workers 8 --out results/frontier-s0.json | tail -3

In [ ]:
!python run_local.py --mode undo --seed 0 --max-actions 10000 --workers 8 --out results/undo-s0.json | tail -3

## Results table (official formula, 25 public games)

In [ ]:
import json
rows = {m: json.load(open(f"results/{m}-s0.json")) for m in ("random", "frontier", "undo")}
games = [g["game"] for g in rows["undo"]["games"]]
print(f"{'game':6s} " + " ".join(f"{m:>16s}" for m in rows))
for i, g in enumerate(games):
    print(f"{g:6s} " + " ".join(f"{r['games'][i]['levels']}/{r['games'][i]['win_levels']} lv {r['games'][i]['score']:6.2f}".rjust(16) for r in rows.values()))
print("-" * 60)
for m, r in rows.items():
    print(f"{m:8s} score {r['total']:6.2f}   levels {sum(g['levels'] for g in r['games'])}/{sum(g['win_levels'] for g in r['games'])}   "
          f"games with >=1 level {sum(g['levels'] > 0 for g in r['games'])}/25")

## What the agent learned about reversibility

Final classification of every tested transition across all levels reached (undo mode, seed 0).

In [ ]:
tot = {}
for g in rows["undo"]["games"]:
    for k, v in g["stats"].items():
        tot[k] = tot.get(k, 0) + v
edges = sum(tot[k] for k in ("reversible_edges", "irreversible_edges", "noop_edges", "fatal_edges"))
for k in ("reversible_edges", "irreversible_edges", "noop_edges", "fatal_edges"):
    print(f"{k:20s} {tot[k]:7d}  {100 * tot[k] / edges:5.1f}%")
print("states visited", tot["states"], " game overs", tot["game_overs"], " resets", tot["resets"])
print("games where ACTION7 was verified as an undo:", [g["game"] for g in rows["undo"]["games"] if g["undo_works"]])
print("games with learned inverse pairs:", {g["game"]: g["inverse_pairs"] for g in rows["undo"]["games"] if g["inverse_pairs"]})

## Kaggle competition rerun hook

This mirrors the official [ARC-AGI-3-Kaggle-Starter](https://github.com/arcprize/ARC-AGI-3-Kaggle-Starter) build:
during a competition rerun the framework's gateway sidecar drives the agent and produces `submission.parquet`.
It is a no-op outside Kaggle. The `UndoAgent` class implements the starter's `is_done` / `choose_action` interface.

In [ ]:
import os, shutil, subprocess, sys, pandas as pd
if os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    fw = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"
    shutil.copytree(fw, "/kaggle/working/framework", dirs_exist_ok=True)
    shutil.copy("undo_agent.py", "/kaggle/working/framework/agents/undo_agent.py")
    # register: agents/__init__.py must export the class under the name passed to --agent
    with open("/kaggle/working/framework/agents/__init__.py", "a") as f:
        f.write("\nfrom .undo_agent import UndoAgent as undoagent\n")
    subprocess.run([sys.executable, "main.py", "--agent", "undoagent"], cwd="/kaggle/working/framework", check=True)
elif not os.path.exists("/kaggle/working/submission.parquet") and os.path.isdir("/kaggle/working"):
    pd.DataFrame(columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet("/kaggle/working/submission.parquet")